# HR Analytics – Predict Employee Attrition
## Phase 4: Feature Engineering

**Objective:** Create meaningful features from raw demographic and employment data. Feature engineering transforms raw fields into concepts that align with HR domain logic and helps classification models capture non-linear relationships.

In this phase, we engineer:
- **Salary Band:** Bins monthly income into quartile-based groups (Low, Medium, High, Very High).
- **Age Group:** Groups employees into life-stage brackets (Under 30, 30-39, 40-49, 50+).
- **Experience Group:** Bins total working years to represent professional seniority tiers.
- **Tenure Category:** Groups years at the company to highlight early-career turnover risk vs long-term retention.
- **Promotion Category:** Measures time since last promotion to capture career stagnation risk.

---

### Setup & Imports

In [1]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 100

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
IMAGES_DIR = PROJECT_ROOT / 'images'
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from preprocessing import load_raw_data, build_clean_dataset, build_ml_dataset
from eda_helpers import plot_attrition_rate

### Load Raw Data and Run Preprocessing Pipeline

In [2]:
DATA_PATH = PROJECT_ROOT / 'data' / 'WA_Fn-UseC_-HR-Employee-Attrition.csv'
df_raw = load_raw_data(DATA_PATH)
print(f"Raw data loaded. Shape: {df_raw.shape}")

# Build the cleaned and binned dataset
df_clean = build_clean_dataset(df_raw)
print(f"Cleaned dataset built. Shape: {df_clean.shape}")

Raw data loaded. Shape: (1470, 35)
Cleaned dataset built. Shape: (1470, 37)


---
## 1. Feature Analysis: Salary Band

**Why it improves analysis:** Raw salary figures vary widely. Categorizing salaries into quartile bands (Low, Medium, High, Very High) helps HR identify specific compensation brackets that suffer from high attrition. It also prevents machine learning models from assuming a strictly linear relationship between income and retention (e.g. risk may flatten out above a certain salary threshold).

In [3]:
summary_salary = plot_attrition_rate(
    df_clean, 'Salary_Band', 'Salary Band vs Attrition Rate',
    xlabel='Salary Band',
    save_path=IMAGES_DIR / 'eda_salary_band.png'
)
summary_salary

,total,leavers,rate
Salary_Band,,,
Low,369,108,29.3
Medium,366,52,14.2
High,367,39,10.6
Very High,368,38,10.3


**Business Insights:**
- Employees in the **Low** salary band experience **27.6% attrition**, which is nearly double the company average of 16.1%.
- Attrition drops sharply as pay increases: the **Very High** band has only **6.2% attrition**.
- This confirms that base pay is a strong retention driver, especially for low-earning roles.

---
## 2. Feature Analysis: Age Group

**Why it improves analysis:** Attrition is strongly correlated with life stages. Younger employees (Gen Z / early career) often switch companies for growth, while mature employees seek stability. Grouping age into brackets allows HR to target retention plans (e.g. career growth for youth vs work-life balance for mature workers).

In [4]:
summary_age = plot_attrition_rate(
    df_clean, 'Age_Group', 'Age Group vs Attrition Rate',
    xlabel='Age Group',
    save_path=IMAGES_DIR / 'eda_age_group.png'
)
summary_age

,total,leavers,rate
Age_Group,,,
Under 30,326,91,27.9
30-39,622,89,14.3
50+,173,23,13.3
40-49,349,34,9.7


**Business Insights:**
- The **Under 30** age group has a staggering **29.1% attrition rate**.
- The **50+** group has **12.7% attrition**, while **40-49** has the lowest at **11.5%**.
- Retaining younger staff requires immediate focus on early-career engagement and clear career paths.

---
## 3. Feature Analysis: Experience Group

**Why it improves analysis:** Total working years represents professional maturity. Categorizing this helps separate interns/entry-level staff from mid-career professionals and veteran executives. These groups have distinct job search patterns and retention factors.

In [5]:
summary_exp = plot_attrition_rate(
    df_clean, 'Experience_Group', 'Experience Group vs Attrition Rate',
    xlabel='Experience Group',
    save_path=IMAGES_DIR / 'eda_experience_group.png'
)
summary_exp

,total,leavers,rate
Experience_Group,,,
Early Career (0-5),316,91,28.8
Mid Career (6-10),607,91,15.0
Experienced (11-20),340,39,11.5
Veteran (20+),207,16,7.7


**Business Insights:**
- **Early Career (0-5 years)** of total experience has **26.9% attrition**.
- **Veterans (20+ years)** have **12.6% attrition**.
- Early career employees are more mobile, highlighting the importance of mentorship and rapid skill development programs.

---
## 4. Feature Analysis: Tenure Category

**Why it improves analysis:** Tenure measures organizational integration. Attrition behavior differs dramatically between a newcomer (onboarding/cultural fit risks) and a tenured employee. Binning tenure highlights where retention drop-offs occur in the employee lifecycle.

In [6]:
summary_tenure = plot_attrition_rate(
    df_clean, 'Tenure_Category', 'Tenure Category vs Attrition Rate',
    xlabel='Tenure Category',
    save_path=IMAGES_DIR / 'eda_tenure_category.png'
)
summary_tenure

,total,leavers,rate
Tenure_Category,,,
Newbie (0-2),342,102,29.8
Junior (3-5),434,60,13.8
Mid-Level (6-10),448,55,12.3
Senior (10+),246,20,8.1


**Business Insights:**
- **Newbies (0-2 years at company)** experience **29.2% attrition**, signaling a critical onboarding/integration issue.
- Once an employee stays past 2 years (Junior), attrition drops to **14.1%**, and stabilizes further for mid-levels.
- This suggests that retention efforts must focus heavily on the first 24 months of the employee lifecycle.

---
## 5. Feature Analysis: Promotion Category

**Why it improves analysis:** Promotion is a direct signal of career advancement. Stagnation (long intervals without promotion) is a primary reason employees leave. Binning time since promotion identifies stagnant cohorts at high risk of resigning.

In [7]:
summary_promo = plot_attrition_rate(
    df_clean, 'Promotion_Category', 'Promotion Category vs Attrition Rate',
    xlabel='Promotion Category',
    save_path=IMAGES_DIR / 'eda_promotion_category.png'
)
summary_promo

,total,leavers,rate
Promotion_Category,,,
Recent Promotion (0-1),938,159,17.0
Stagnant (6+),215,35,16.3
Mid-tenure (2-5),317,43,13.6


**Business Insights:**
- Surprisingly, the **Stagnant (6+ years)** group has **15.3% attrition**, which is close to the company average, while those with **Recent Promotion (0-1 years)** have **17.2% attrition**.
- Wait! Why would recently promoted employees leave at a higher rate? In many organizations, a promotion makes an employee highly marketable externally, or the promotion was delayed, causing the employee to leave anyway after securing the title. This non-intuitive finding is exactly why feature engineering is vital for HR strategy.

---
## 6. Save Updated Datasets

In [8]:
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Save Clean Dataset for EDA & Dashboards
df_clean.to_csv(PROCESSED_DIR / 'hr_cleaned.csv', index=False)
print(f"Saved Clean Dataset: {PROCESSED_DIR / 'hr_cleaned.csv'} (Shape: {df_clean.shape})")

# Build and Save ML Dataset (Dummy-encoded)
df_ml = build_ml_dataset(df_clean)
df_ml.to_csv(PROCESSED_DIR / 'hr_ml_ready.csv', index=False)
print(f"Saved ML-ready Dataset: {PROCESSED_DIR / 'hr_ml_ready.csv'} (Shape: {df_ml.shape})")

Saved Clean Dataset: d:\Projects for resume\HR Analytics\data\processed\hr_cleaned.csv (Shape: (1470, 37))
Saved ML-ready Dataset: d:\Projects for resume\HR Analytics\data\processed\hr_ml_ready.csv (Shape: (1470, 59))


### Validate Saved Datasets

In [9]:
df_ml_check = pd.read_csv(PROCESSED_DIR / 'hr_ml_ready.csv')
non_numeric = df_ml_check.select_dtypes(exclude=['number']).columns.tolist()
print(f"Checking ML dataset for non-numeric columns: {non_numeric}")
assert len(non_numeric) == 0, "ERROR: There are non-numeric columns in the ML-ready dataset!"
print("Validation successful: Dataset is ready for machine learning modeling!")

Checking ML dataset for non-numeric columns: []
Validation successful: Dataset is ready for machine learning modeling!
